# End-to-End AutoML for Telecom Customer Churn

## Part 1 - EDA and Data Pre-Processing

### Contents
[Part 1 - Project Overview](#overview)  
[Part 2 - Initial Setup](#setup)  
[Part 3 - Exploratory Data Analysis](#eda)  
[Part 4 - Data Pre-Processing and Transformation](#pre-processing)  
[Part 5 - Export for the AutoML stack](#export)  
[Part 6 - References](#references)

___
<a name="overview"></a>
## (1) Project Overview

**The Challenge**  
- Telecom Churn Model: predict which subscribers are about to cancel (churn) their plan.
- This is a binary classification task. Output is `1` (will churn) or `0` (will stay).

Link: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
___
**The Data**  
Each row is one customer of a telecom company. Key columns:

**customerID:** unique ID for the customer (dropped, not a predictor)  
**gender, SeniorCitizen, Partner, Dependents:** demographics  
**tenure:** number of months the customer has stayed  
**PhoneService, MultipleLines, InternetService:** subscribed services  
**OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport:** add-ons  
**StreamingTV, StreamingMovies:** streaming options  
**Contract, PaperlessBilling, PaymentMethod:** billing details  
**MonthlyCharges, TotalCharges:** money paid  
**Churn (target):** Yes = customer left, No = customer stayed

___
<a name="setup"></a>
## (2) Initial Setup

### Import dependencies

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import requests
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Folder paths (relative to /home/jovyan/work inside the container)
RAW_DIR = os.path.join('..', 'data', 'raw')
OUT_DIR = os.path.join('..', 'output')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

RAW_CSV = os.path.join(RAW_DIR, 'telco_churn.csv')
print('Raw file target :', os.path.abspath(RAW_CSV))
print('Output folder   :', os.path.abspath(OUT_DIR))

### Download Dataset
We download the public IBM Telco Customer Churn CSV with `requests`. If the container has no internet, see the manual fallback in the cell below.

In [ ]:
URL = (
    'https://raw.githubusercontent.com/IBM/'
    'telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
)

if os.path.exists(RAW_CSV):
    print('Raw file already present - skipping download.')
else:
    try:
        print('Downloading from:', URL)
        resp = requests.get(URL, timeout=60)
        resp.raise_for_status()
        with open(RAW_CSV, 'wb') as f:
            f.write(resp.content)
        print('Download OK. Size:', len(resp.content), 'bytes')
    except Exception as exc:
        print('Download FAILED:', exc)
        print('Use the manual fallback below, then re-run all cells.')

**Manual fallback (only if the download failed).** Run this on your machine (host), from the `preprocessing-telecom-churn` folder, then re-run all cells:

```powershell
Invoke-WebRequest -Uri "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv" -OutFile "data\raw\telco_churn.csv"
```

Or download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), copy it to `data/raw/`, and rename it `telco_churn.csv`.

### Import Data

In [ ]:
main_df = pd.read_csv(RAW_CSV)
print('Rows:', len(main_df))
main_df.head()

### Setup helper functions

In [ ]:
# Display value counts with percentages
def show_value_counts(df, column):
    out = pd.DataFrame(df[column].value_counts().rename_axis(column).reset_index(name='counts'))
    out['percentage'] = round(100 * (out['counts'] / len(df)), 1)
    return out

# Display NaN counts with percentages
def show_nan_counts(df):
    out = pd.DataFrame(df.isna().sum()).sort_values(by=0, ascending=False)
    out.columns = ['counts']
    out['percentage'] = round(100 * (out['counts'] / len(df)), 1)
    return out

# Display correlation heatmap of numerical columns
def show_heatmap(df):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
    plt.title('Correlation heatmap (numerical columns)')
    plt.show()

___
<a name="eda"></a>
## (3) Exploratory Data Analysis (EDA)

In [ ]:
main_df.shape

In [ ]:
main_df.sample(10)

___
### Churn (target) variable

In [ ]:
show_value_counts(main_df, 'Churn')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
sns.countplot(x=main_df['Churn'], order=['No', 'Yes'], ax=ax)
plt.title('Churn class balance')
plt.show()

The classes are imbalanced (around 73% No, 27% Yes). H2O AutoML handles this automatically via `balance_classes=True` in the main training script.

### Predictor Variables

In [ ]:
main_df.dtypes

In [ ]:
# Identify columns which are not numerical
categorical = [col for col in main_df.columns if main_df[col].dtype == 'object']
categorical

In [ ]:
show_nan_counts(main_df)

Notice `TotalCharges` is an `object` (text) even though it should be numbers. It contains blank spaces for brand-new customers (tenure 0). We fix this in pre-processing.

In [ ]:
# Distribution of the main numerical features
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(main_df['tenure'], kde=True, ax=axes[0]).set_title('tenure (months)')
sns.histplot(main_df['MonthlyCharges'], kde=True, ax=axes[1]).set_title('MonthlyCharges')
plt.show()

In [ ]:
# Churn rate by contract type - a strong predictor
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=main_df, x='Contract', hue='Churn', ax=ax)
plt.title('Churn by contract type')
plt.show()

___
<a name="pre-processing"></a>
## (4) Data Pre-Processing and Transformation

In [ ]:
main_df.head()

In [ ]:
def preprocess_data(df):
    df = df.copy()

    # 1) Drop the identifier column (not a predictor)
    if 'customerID' in df.columns:
        df = df.drop(columns=['customerID'])

    # 2) Fix TotalCharges: blanks -> NaN -> numeric -> fill missing with 0
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

    # 3) Encode the target Churn: Yes -> 1, No -> 0
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

    # 4) One-hot encode all remaining categorical (object) columns
    cat_cols = [c for c in df.columns if df[c].dtype == 'object']
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    return df

In [ ]:
main_df = preprocess_data(main_df)
print('Shape after preprocessing:', main_df.shape)
main_df.head()

### Train Test Split
We keep 90% for training and 10% for the demo test sample, stratified on `Churn` to keep the same class balance in both.

In [ ]:
TARGET = 'Churn'

train_df, test_df = train_test_split(
    main_df,
    test_size=0.10,
    random_state=42,
    stratify=main_df[TARGET],
)
print('train_df:', train_df.shape)
print('test_df :', test_df.shape)

### Scale Data
We MinMax-scale the three numerical features. The scaler is fit on the train set and reused (loaded from pickle) on the test set, exactly like the original insurance notebook.

In [ ]:
NUMERICAL = ['tenure', 'MonthlyCharges', 'TotalCharges']
SCALER_PATH = os.path.join(OUT_DIR, 'minmax_scaler.pkl')

def scale_data(df, train):
    df = df.copy()
    if train:
        mm = MinMaxScaler()
        df[NUMERICAL] = mm.fit_transform(df[NUMERICAL])
        pickle.dump(mm, open(SCALER_PATH, 'wb'))
    else:
        mm = pickle.load(open(SCALER_PATH, 'rb'))
        df[NUMERICAL] = mm.transform(df[NUMERICAL])
    return df

In [ ]:
train_df = scale_data(train_df, train=True)
test_df = scale_data(test_df, train=False)

# Both sets must have the exact same columns
assert len(train_df.columns) == len(test_df.columns)
train_df.head()

___
<a name="export"></a>
## (5) Export for the AutoML stack
We build a small demo sample (50 rows) and write the three files the main project needs.

In [ ]:
# Small demo sample from the test set
SAMPLE_SIZE = 50
sample_labeled = test_df.sample(n=min(SAMPLE_SIZE, len(test_df)), random_state=42)
sample_nolabel = sample_labeled.drop(columns=[TARGET])

train_path   = os.path.join(OUT_DIR, 'train.csv')
test_path    = os.path.join(OUT_DIR, 'sample_test.csv')
labeled_path = os.path.join(OUT_DIR, 'sample_test_labeled.csv')

train_df.to_csv(train_path, index=False)
sample_nolabel.to_csv(test_path, index=False)
sample_labeled.to_csv(labeled_path, index=False)

print('Saved:')
print(' -', os.path.abspath(train_path))
print(' -', os.path.abspath(test_path))
print(' -', os.path.abspath(labeled_path))

In [ ]:
# Final sanity checks
for p in [train_path, test_path, labeled_path]:
    cols = pd.read_csv(p, nrows=1).columns.tolist()
    print(f'{os.path.basename(p):28s} has_Churn={"Churn" in cols}  n_cols={len(cols)}')

print()
print('Expected -> train.csv: True | sample_test.csv: False | sample_test_labeled.csv: True')

### Done! Copy the files into the main project

| From | To |
|---|---|
| `output/train.csv` | `backend/data/processed/train.csv` |
| `output/sample_test.csv` | `backend/data/sample_test.csv` |
| `output/sample_test_labeled.csv` | `backend/data/sample_test_labeled.csv` |

Then in the main project set `--target Churn`, `MODEL_NAME: telecom-churn-automl`, update `TARGET_COL`/`LABELS` in `frontend/app.py`, and run `docker compose up --build`.

___
<a name="references"></a>
## (6) References
- Telco Customer Churn dataset (Kaggle): https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- IBM sample data (raw CSV used for download): https://github.com/IBM/telco-customer-churn-on-icp4d
- scikit-learn MinMaxScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html